# Dropping Missing Values - When Method 1 Actually Fires

`loans.csv` never had a column over 50% missing, so Method 1 from `06_Dropping_Missing_Values.MD` never removed anything. This notebook uses a new dataset, `loans_guarantor.csv`, built specifically to trigger it.

**What's different:** same 618 applicants and the same 13 original columns as `loans.csv`, plus one new column - `Guarantor_Income` - which is only filled in when an applicant actually has a guarantor. Most don't, so it's genuinely, realistically missing most of the time.

In [1]:
import pandas as pd

In [2]:
dataset = pd.read_csv("loans_guarantor.csv")
dataset.shape

(618, 14)

In [3]:
dataset.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status,Guarantor_Income
0,LP001001,Male,Yes,0,Not Graduate,No,1686.0,1020,80.0,360.0,1.0,Semiurban,Y,NaN
1,LP001002,Male,Yes,0,Not Graduate,No,2437.0,1201,122.0,360.0,1.0,Urban,Y,1935.0
2,LP001003,Female,No,0,Graduate,No,8313.0,1065,319.0,360.0,1.0,Urban,Y,965.0
3,LP001004,Male,Yes,0,Graduate,No,5069.0,0,218.0,360.0,1.0,Semiurban,Y,NaN
4,LP001005,Male,No,2,Graduate,No,3002.0,2283,194.0,360.0,0.0,Urban,N,NaN


14 columns now instead of 13 - `Guarantor_Income` is the new one, `NaN` for every applicant without a guarantor.

In [5]:
dataset.isnull().sum()

Loan_ID                0
Gender                13
Married                6
Dependents            15
Education              9
Self_Employed         32
ApplicantIncome        2
CoapplicantIncome      0
LoanAmount            22
Loan_Amount_Term      21
Credit_History        50
Property_Area          9
Loan_Status            0
Guarantor_Income     382
dtype: int64

In [6]:
missing_pct = (dataset.isnull().sum() / dataset.shape[0]) * 100
missing_pct.round(2)

Loan_ID               0.00
Gender                2.10
Married               0.97
Dependents            2.43
Education             1.46
Self_Employed         5.18
ApplicantIncome       0.32
CoapplicantIncome     0.00
LoanAmount            3.56
Loan_Amount_Term      3.40
Credit_History        8.09
Property_Area         1.46
Loan_Status           0.00
Guarantor_Income     61.81
dtype: float64

`Guarantor_Income` sits at roughly **62% missing** - well past the 50% line. First time in this repo's practice that Method 1 has something real to do.

## Method 1 - does it fire this time?

In [7]:
missing_pct >= 50

Loan_ID              False
Gender               False
Married              False
Dependents           False
Education            False
Self_Employed        False
ApplicantIncome      False
CoapplicantIncome    False
LoanAmount           False
Loan_Amount_Term     False
Credit_History       False
Property_Area        False
Loan_Status          False
Guarantor_Income      True
dtype: bool

This time the mask isn't all `False` - `Guarantor_Income` is `True`.

In [8]:
missing_pct[missing_pct >= 50]

Guarantor_Income    61.812298
dtype: float64

In [9]:
cols_to_drop = missing_pct[missing_pct >= 50].index
cols_to_drop

Index(['Guarantor_Income'], dtype='str')

Not an empty `Index` this time - it actually contains `"Guarantor_Income"`.

In [10]:
dataset_cleaned = dataset.drop(columns=cols_to_drop)
dataset_cleaned.shape

(618, 13)

14 columns -> 13. `Guarantor_Income` is gone, everything else untouched.

## Why column-first matters - now with real numbers

`06_Dropping_Missing_Values.MD` argued that skipping Method 1 and going straight to row-dropping is risky. `loans.csv` couldn't prove that, since nothing there crossed 50%. This dataset can.

In [11]:
dataset.dropna().shape

(184, 14)

In [11]:
dataset.shape[0] - dataset.dropna().shape[0]

434

Blind `dropna()` on the **raw, 14-column** dataset - before removing the bad column - loses this many rows. `Guarantor_Income` alone drags down hundreds of otherwise-fine applicants.

In [12]:
dataset_cleaned.dropna().shape

(457, 13)

In [13]:
dataset_cleaned.shape[0] - dataset_cleaned.dropna().shape[0]

161

**Two very different answers depending on order:**

| Order | Rows lost |
|---|---|
| `dropna()` on the raw 14 columns (no Method 1 first) | ~434 rows (~70%) |
| Method 1 first (drop `Guarantor_Income`), *then* `dropna()` | 161 rows (~26%) |

One bad column, left in, nearly triples the damage. This is the concrete version of `06_Dropping_Missing_Values.MD`'s "why column-first" argument.

## Before dropping it outright - the missing-indicator alternative

`06_Dropping_Missing_Values.MD`'s caveat about `Credit_History` applies here too: *not having a guarantor* might itself be predictive of loan risk, even though the exact income figure is unknown. Dropping the column entirely throws that signal away. A middle path: keep a flag for "was this ever filled in," drop only the raw (mostly-empty) number.

In [12]:
dataset["Had_Guarantor"] = dataset["Guarantor_Income"].notnull().astype(int)
dataset["Had_Guarantor"].value_counts()

Had_Guarantor
0    382
1    236
Name: count, dtype: int64

`.notnull()` flips the missingness around - `True` where a value *exists*. `.astype(int)` turns that into a clean `1`/`0` column instead of `True`/`False`.

In [13]:
dataset_with_flag = dataset.drop(columns=["Guarantor_Income"])
dataset_with_flag.shape

(618, 14)

Same column *count* as `dataset_cleaned`, but not the same columns - this version keeps `Had_Guarantor` (the signal) and only drops the noisy raw income figure, instead of losing the guarantor information completely.

## Recap

- `loans_guarantor.csv` = `loans.csv` + one new column, `Guarantor_Income`, at ~62% missing - deliberately built to cross the 50% line.
- **Method 1 fires for real here**: `cols_to_drop` actually contains a column, and `dataset_cleaned` drops from 14 to 13 columns.
- **Order matters, concretely**: skipping Method 1 costs ~434 rows (70%); doing Method 1 first costs only 161 rows (26%) - the same 161 `loans.csv` already showed in `06_Dropping_Missing_Values.ipynb`, since the other 12 columns are identical.
- **`dataset_with_flag`** is the more careful alternative from the `.MD` file's caveat - keep the *fact* that it was missing as its own feature, instead of deleting that information along with the column.